# 🏪 Zava Agentic Fine-Tuning Lab — 04: Build Data & Submit a Job

**In this notebook**, you'll:
1. Understand the RFT training data format
2. Build your own training example from scratch
3. Upload training/validation files and submit a real RFT job

| What you'll do | Time |
|----------------|------|
| Explore the data format | 2 min |
| Build a custom training example | 5 min |
| Upload files and submit RFT job | 8 min |

> **Prerequisite**: Complete `01-introduction-setup.ipynb` first.

---
## Setup — Reconnect and Load Agent Infrastructure

We need the client, tools, and grader to test our training examples.

In [1]:
import json, os, re, time, textwrap
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)
print("✅ Connected to Azure AI Foundry")

✅ Connected to Azure AI Foundry


In [2]:
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "function_app"))
from function_app import get_order as _get_order_local

# === Tool endpoint (pre-deployed Azure Function) ===
TOOL_URL = "https://zava-rft-tools-bethany.azurewebsites.net"

# The system prompt the agent uses
SYSTEM_PROMPT = """You are Zava's return resolution engine. Call get_order to look up order details, then apply the return policy to compute the resolution.

POLICY: Standard=30d/15d(electronics), Gold=45d/30d, Platinum=60d/45d. Electronics restocking: Std=15%, Gold=7.5%, Plat=0%. Defective=0%. Sale=final sale (defective sale→store credit). Late delivery(>2d)=$10 credit +15d extension. Lost=replacement/refund. Pending=cancellable. Opened personal care=deny unless defective.

Respond with your resolution including: action, amounts, and policy reasoning."""

# Tool definition (same schema the model sees)
TOOLS = [
    {"type": "function", "function": {
        "name": "get_order",
        "description": "Look up order details including items, prices, dates, loyalty tier, and delivery status.",
        "parameters": {"type": "object", "properties": {
            "order_id": {"type": "string", "description": "The order ID (e.g., ORD-003)"}
        }, "required": ["order_id"]}
    }}
]


def call_tool(name, args):
    """Call the Zava tool endpoint, falling back to local implementation if unavailable."""
    if name == "get_order":
        try:
            url = f"{TOOL_URL}/tool/{name}"
            payload = {"arguments": json.dumps(args), "call_id": "c", "id": "f", "trace_id": "t"}
            r = requests.post(url, json=payload, timeout=5)
            if r.status_code == 200:
                return r.json().get("output", json.dumps(r.json()))
        except Exception:
            pass
        return _get_order_local(args["order_id"])
    return json.dumps({"error": f"Unknown tool: {name}"})


def run_agent(user_message, model="o4-mini", verbose=True):
    """Run the full agent loop: model → tool call → model → response."""
    messages = [
        {"role": "developer", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    tool_calls_made = []

    for turn in range(8):
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, max_completion_tokens=8192
        )
        msg = resp.choices[0].message

        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
            tool_calls_made.extend(msg.tool_calls)
        messages.append(assistant_msg)

        if not msg.tool_calls:
            if verbose and msg.content:
                print(f"\n📋 Agent Response:\n{textwrap.fill(msg.content, width=80)}")
            return msg.content or "", tool_calls_made

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"  🔧 Calling {tc.function.name}({args})")
            result = call_tool(tc.function.name, args)
            if verbose:
                preview = result[:200] + "..." if len(result) > 200 else result
                print(f"  📦 Result: {preview}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "", tool_calls_made


def python_grader(output_text, output_tools, expected_resolution):
    """Score a model response against the expected resolution.
    Returns 0.0 to 1.0 — same logic used during RFT training."""
    if not expected_resolution:
        return 0.5

    score = 0.0
    exp_lower = expected_resolution.lower()
    out_lower = (output_text or "").lower()

    # Action correctness (0.4)
    actions = {
        "refund": ["refund"],
        "denied": ["denied", "deny", "not eligible", "cannot", "expired"],
        "store credit": ["store credit", "store_credit"],
        "replacement": ["replacement", "replace"],
        "exchange": ["exchange", "swap"],
        "cancel": ["cancel", "cancellation"],
    }
    for action, keywords in actions.items():
        if any(k in exp_lower for k in keywords):
            if any(k in out_lower for k in keywords):
                score += 0.4
            break

    # Amount correctness (0.3)
    exp_amounts = re.findall(r'\$(\d+\.\d{2})', expected_resolution)
    if exp_amounts:
        out_amounts = re.findall(r'\$(\d+\.\d{2})', output_text or "")
        hits = sum(1 for a in exp_amounts if a in out_amounts)
        score += 0.3 * (hits / len(exp_amounts))
    else:
        score += 0.15

    # Policy reasoning (0.2)
    policy_terms = ["window", "restocking", "defective", "sale", "platinum", "gold",
                    "standard", "late", "shipping credit", "personal care", "eligible"]
    exp_terms = [t for t in policy_terms if t in exp_lower]
    if exp_terms:
        hits = sum(1 for t in exp_terms if t in out_lower)
        score += 0.2 * (hits / len(exp_terms))

    # Tool usage bonus (0.1)
    if output_tools:
        tool_names = [t.function.name if hasattr(t, 'function') else t.get("function", {}).get("name", "") for t in output_tools]
        if "get_order" in tool_names:
            score += 0.1

    return round(min(score, 1.0), 3)

print("✅ Agent + grader ready")

✅ Agent + grader ready


---
## Section 5: Build Training Data (5 min)

RFT training data is simpler than SFT — you only need **prompts** (not ideal responses).
The model generates its own responses during training, and the grader scores them.

Each example needs:
- `messages` — the conversation (developer prompt + user request)
- `expected_resolution` — the correct answer, used by the grader to score

**You don't write model responses!** The model figures those out through trial and error.

### Explore the Data Format

In [3]:
# Let's look at the data format
sample = json.loads(open("data/rft_v7_train.jsonl").readline())
print("Training example format:")
print(json.dumps(sample, indent=2)[:600])

Training example format:
{
  "messages": [
    {
      "role": "system",
      "content": "You are Zava's Post-Purchase Resolution Desk agent. You help customers with returns, exchanges, replacements, cancellations, and shipping disputes.\n\n## Available Tools (call in this order)\n1. **get_order_details** - Retrieve order info, line items, customer loyalty tier\n2. **get_fulfillment_status** - Check delivery status, late delivery, lost packages\n3. **check_resolution_policy** - Verify eligibility per item (call once PER item)\n4. **check_inventory** - Check stock ONLY when processing an exchange\n5. **calculate_resol


### Quick dataset overview

Let's see how many training and validation examples we have, and what they look like.

In [4]:
# Dataset overview
with open("data/rft_v7_train.jsonl") as f:
    train_data = [json.loads(line) for line in f]
with open("data/rft_v7_val.jsonl") as f:
    val_data = [json.loads(line) for line in f]

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
print(f"\nSample user messages from training set:")
for i, ex in enumerate(train_data[:5]):
    user_msg = ex["messages"][-1]["content"]
    print(f"  [{i+1}] {user_msg[:80]}..." if len(user_msg) > 80 else f"  [{i+1}] {user_msg}")

Training examples: 43
Validation examples: 8

Sample user messages from training set:
  [1] Hi, I'm Sofia Martinez (sofia.martinez@example.com). I ordered a Merino Wool Swe...
  [2] Hi, I'm Noah Brown (noah.brown@example.com). Can you check the status of my orde...
  [3] I'm Ava Chen, ava.chen@example.com. My order ORD-006 for a yoga mat never arrive...
  [4] Hi, I'm Yusuf Rossi (yusuf.rossi@example.com). I placed order ORD-012 yesterday ...
  [5] Hello, I'm Emma Kim (emma.kim@example.com). The LED Desk Lamp from my order ORD-...


### Build Your Own Example

Let's create a training example from scratch. First, call the tool to see what
order data is available, then write the expected resolution.

In [5]:
# Look up an order to understand the data
order_data = call_tool("get_order", {"order_id": "ORD-003"})
print("Order ORD-003 data:")
print(json.dumps(json.loads(order_data), indent=2))

Order ORD-003 data:
{
  "order_id": "ORD-003",
  "customer": {
    "id": "C003",
    "name": "Yusuf Rossi",
    "email": "yusuf.rossi@example.com",
    "loyalty_tier": "gold"
  },
  "order_date": "2026-06-18",
  "promised_delivery": "2026-06-23",
  "items": [
    {
      "item_id": "LI-003",
      "product_id": "P002",
      "product_name": "Mechanical Keyboard",
      "category": "electronics",
      "sku": "P002-BLK",
      "quantity": 1,
      "unit_price": 129.99,
      "discount_pct": 0,
      "on_sale": false,
      "variant": ""
    },
    {
      "item_id": "LI-004",
      "product_id": "P013",
      "product_name": "Urban Sneakers",
      "category": "apparel",
      "sku": "P013",
      "quantity": 1,
      "unit_price": 64.99,
      "discount_pct": 0,
      "on_sale": true,
      "variant": ""
    }
  ],
  "fulfillment": [
    {
      "item_id": "LI-003",
      "status": "delivered",
      "ship_date": "2026-06-21",
      "delivery_date": "2026-06-25",
      "carrier": "FedE

In [6]:
# Now write a training example. ORD-003 belongs to Yusuf Rossi (Gold tier).
# Think: what's a realistic customer request? What's the correct policy resolution?

my_example = {
    "messages": [
        {"role": "developer", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Yusuf Rossi here. The keyboard from ORD-003 stopped working after a week. Keys are unresponsive."}
    ],
    "expected_resolution": "Refund $89.99 for defective keyboard. Gold tier, within 45-day window. Defective items have $0 restocking fee."
}

# Test: what does base o4-mini say for our example?
output, tools = run_agent(my_example["messages"][-1]["content"], verbose=True)
score = python_grader(output, tools, my_example["expected_resolution"])
print(f"\n🎯 Grader score: {score:.3f}")

  🔧 Calling get_order({'order_id': 'ORD-003'})
  📦 Result: {
  "order_id": "ORD-003",
  "customer": {
    "id": "C003",
    "name": "Yusuf Rossi",
    "email": "yusuf.rossi@example.com",
    "loyalty_tier": "gold"
  },
  "order_date": "2026-06-18",
  "promise...

📋 Agent Response:
Resolution for order ORD-003 (Mechanical Keyboard):  Action   • Full refund of
the keyboard purchase price plus associated sales tax.     – Item price: $129.99
– Sales tax (allocated): $10.40     – Total refund: $140.39    Policy reasoning
• Loyalty tier: Gold (30-day return window for electronics).   • Delivery was on
2026-06-25; today is 2026-07-15 (20 days since delivery), so the return is
within your 30-day electronics window.   • Item is defective (stopped working),
which qualifies for a full refund with no restocking fee.    Next steps   Your
refund (total $140.39) will be issued to your original debit card within 3–5
business days. If you would prefer a replacement unit instead of a refund,
please let 

> **💡 Key insight**: For RFT, data diversity matters more than volume.
> We wrote ~40 unique scenarios by hand, then used an LLM to generate 10 rephrasings
> of each (different tones, wordings, levels of detail). That gives 400 examples that
> teach the model to handle the same policy rules with varied customer phrasings.
>
> Our pre-built dataset (`rft_v7_train.jsonl`) has 343 examples built this way.

---
## Section 6: Submit an RFT Job (10 min)

Now let's submit a real RFT training job with the pre-built dataset.
The process has 4 steps:
1. Upload training and validation files
2. Wait for file processing
3. Define the grader (Python source code)
4. Submit the fine-tuning job

### Step 1: Upload Files

In [7]:
# Upload training and validation files
print("Uploading training data...")
with open("data/rft_v7_train.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
print(f"  Train: {train_file.id} ({train_file.bytes:,} bytes)")

with open("data/rft_v7_val.jsonl", "rb") as f:
    val_file = client.files.create(file=f, purpose="fine-tune")
print(f"  Val: {val_file.id} ({val_file.bytes:,} bytes)")

# Wait for processing
import time
for _ in range(12):
    t = client.files.retrieve(train_file.id)
    v = client.files.retrieve(val_file.id)
    if t.status == "processed" and v.status == "processed":
        print("  ✅ Files ready!")
        break
    time.sleep(10)

Uploading training data...
  Train: file-2c2f0a7011fa4302b13e05c7d178787d (315,551 bytes)
  Val: file-660b29d5772142138446f6ce88eae323 (9,972 bytes)
  ✅ Files ready!


### Step 2: Define the Grader

The grader is the same Python function we used for local evaluation, but embedded
as a string so the training service can execute it. It must define a `grade(sample, item)`
function that returns a score between 0.0 and 1.0.

In [8]:
# Define the grader (same Python function, embedded as a string)
GRADER_SOURCE = r"""
import json
import re

def grade(sample, item):
    output_text = sample.get("output_text", "") or ""
    output_tools = sample.get("output_tools", []) or []
    expected = item.get("expected_resolution", "")
    if not expected:
        return 0.5
    score = 0.0
    exp_lower = expected.lower()
    out_lower = output_text.lower()

    # Action (0.4)
    actions = {"refund": ["refund"], "denied": ["denied", "deny", "not eligible", "cannot", "expired"],
        "store credit": ["store credit", "store_credit"], "replacement": ["replacement", "replace"],
        "exchange": ["exchange", "swap"], "cancel": ["cancel", "cancellation"]}
    for action, keywords in actions.items():
        if any(k in exp_lower for k in keywords):
            if any(k in out_lower for k in keywords):
                score += 0.4
            break

    # Amount (0.3)
    exp_amounts = re.findall(r'\$(\ d+\.\d{2})', expected)
    if exp_amounts:
        out_amounts = re.findall(r'\$(\d+\.\d{2})', output_text)
        hits = sum(1 for a in exp_amounts if a in out_amounts)
        score += 0.3 * (hits / len(exp_amounts))
    else:
        score += 0.15

    # Policy terms (0.2)
    policy_terms = ["window", "restocking", "defective", "sale", "platinum", "gold",
                    "standard", "late", "shipping credit", "personal care", "eligible"]
    exp_terms = [t for t in policy_terms if t in exp_lower]
    if exp_terms:
        hits = sum(1 for t in exp_terms if t in out_lower)
        score += 0.2 * (hits / len(exp_terms))

    # Tool usage (0.1)
    if output_tools:
        tool_names = [t.get("function", {}).get("name", "") for t in output_tools]
        if "get_order" in tool_names:
            score += 0.1

    return round(min(score, 1.0), 3)
"""

print("✅ Grader source defined")
print(f"   {len(GRADER_SOURCE.strip().splitlines())} lines of grading logic")

✅ Grader source defined
   47 lines of grading logic


### Step 3: Submit the Job!

This creates the RFT fine-tuning job with:
- **Model**: o4-mini (base model to fine-tune)
- **Grader**: Our Python grader with pass_threshold=0.80
- **Tools**: The `get_order` endpoint the model can call during training
- **Hyperparameters**: 3 epochs, medium reasoning effort, checkpoints every 5 steps

In [9]:
# Submit the RFT job
TOOL_CONFIG = [
    {"name": "get_order",
     "server_url": f"{TOOL_URL}/tool/get_order",
     "headers": {}}
]

job = client.fine_tuning.jobs.create(
    model="o4-mini",
    training_file=train_file.id,
    validation_file=val_file.id,
    suffix="zava-lab",
    method={"type": "reinforcement", "reinforcement": {
        "grader": {
            "type": "python",
            "name": "zava_grader",
            "source": GRADER_SOURCE.strip(),
            "pass_threshold": 0.80,        # <-- 35% fail rate = good signal
        },
        "tools": TOOL_CONFIG,
        "max_episode_steps": 5,
        "hyperparameters": {
            "n_epochs": 3,
            "learning_rate_multiplier": 1.0,
            "compute_multiplier": 1.5,
            "reasoning_effort": "medium",
            "eval_interval": 5,
            "eval_samples": 10,
        },
    }},
)

print(f"🚀 Job submitted!")
print(f"   ID: {job.id}")
print(f"   Status: {job.status}")
print(f"   Model: {job.model}")
print(f"\n⏳ Training takes ~2-4 hours. We'll explore pre-run results in the next notebook.")

🚀 Job submitted!
   ID: ftjob-d67e2f3edee641db8661e477e83a17fa
   Status: pending
   Model: o4-mini-2025-04-16

⏳ Training takes ~2-4 hours. We'll explore pre-run results in the next notebook.


---
## 💡 Key Takeaways

- RFT data only needs **prompts + expected answers** — no model responses
- **Data diversity** matters more than volume (40 scenarios × 10 rephrasings = 400 examples)
- The grader must define a `grade(sample, item)` function returning 0.0–1.0
- **pass_threshold** controls the RL reward boundary (0.80 = ~35% failure rate)
- Tools are live during training — the model makes real API calls

**Next → Open `05-training-results-evaluate.ipynb` to explore training metrics and compare models.**